In [1]:
import tweepy

api_key = "o6lygttEU0bsJDzumpFeky5AR"
api_key_secret = "kGK2aUeioUW5gR15pP56MfvC8A3HLJze7c5G5kvLFyPFZHr0sB"
bearer_token = "AAAAAAAAAAAAAAAAAAAAABFAwQEAAAAAYjpL6Fhsbkd0yKw1v15nDMgkUGc%3DDVA4gux4bhgNqdGYm3lYvmOu6yxw0pGEdiWXtGUov2TmrQ7ojB"
access_token = "1843914239230222336-zG9fgt1AzBLfeOZuwlRf0XT14DAl9S"
access_token_secret = "uc6TONQ5B4ZqYEmmbRG0NzIQMFbDWhD20s52zZOpTooUc"

# Autentikasi
auth = tweepy.OAuth1UserHandler(api_key, api_key_secret, access_token, access_token_secret)
api = tweepy.API(auth)

# Cek apakah autentikasi berhasil
try:
    api.verify_credentials()
    print("Autentikasi berhasil")
except:
    print("Autentikasi gagal")


Autentikasi berhasil


In [2]:
from bs4 import BeautifulSoup
import requests

url = "https://x.com/search?q=Shin%20Tae%20Yong&src=typeahead_click"
data = requests.get(url)
scrap = BeautifulSoup(data.text, "html")
print(scrap.prettify())

<!DOCTYPE html>
<head>
 <title>
  x.com
 </title>
 <meta content="0; url = https://twitter.com/x/migrate?tok=7b2265223a222f7365617263683f713d5368696e253230546165253230596f6e67267372633d7479706561686561645f636c69636b222c2274223a313732383631373033337dd330a16cc757272fa169e36bc496bfba" http-equiv="refresh"/>
 <meta charset="utf-8"/>
 <meta content="width=device-width,initial-scale=1,maximum-scale=1,user-scalable=0,viewport-fit=cover" name="viewport"/>
 <link href="//abs.twimg.com" rel="preconnect"/>
 <link href="//abs.twimg.com" rel="dns-prefetch"/>
 <link href="//api.twitter.com" rel="preconnect"/>
 <link href="//api.twitter.com" rel="dns-prefetch"/>
 <link href="//api.x.com" rel="preconnect"/>
 <link href="//api.x.com" rel="dns-prefetch"/>
 <link href="//pbs.twimg.com" rel="preconnect"/>
 <link href="//pbs.twimg.com" rel="dns-prefetch"/>
 <link href="//t.co" rel="preconnect"/>
 <link href="//t.co" rel="dns-prefetch"/>
 <meta content="https://twitter3e4tixl4xyajtrzo62zg5vztmjuricljdp2c5ks

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time

# Pengaturan opsi untuk menjalankan Chrome tanpa antarmuka grafis (headless)
chrome_options = Options()
chrome_options.add_argument("--headless")

# Ganti path dengan lokasi ChromeDriver di komputer kamu
service = Service(executable_path='/Applications/MAMP/htdocs/local/kodingakademi/Python/python 3/machine learning/ML/sentimen_analis/chromedriver')
driver = webdriver.Chrome(service=service, options=chrome_options)

# URL pencarian Twitter untuk Shin Taeyoung
url = "https://x.com/search?q=Shin%20Tae%20Yong&src=recent_search_click"
driver.get(url)

# Tunggu hingga elemen tweet tersedia
try:
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.XPATH, '//div[@data-testid="tweet"]')))
except TimeoutException:
    print("Tweet tidak ditemukan dalam waktu yang ditentukan")

# Tunggu beberapa detik tambahan untuk memastikan konten dimuat
time.sleep(5)

# Mengambil tweet
tweets = driver.find_elements(By.XPATH, '//div[@data-testid="tweet"]')

# Menampilkan hasil
if tweets:
    for tweet in tweets:
        tweet_text = tweet.find_element(By.XPATH, './/div[@lang]').text
        tweet_time = tweet.find_element(By.TAG_NAME, 'time').get_attribute('datetime')
        print(f"Waktu: {tweet_time}")
        print(f"Tweet: {tweet_text}")
        print("-" * 50)
else:
    print("Tidak ada tweet yang ditemukan")

# Menutup driver
driver.quit()


Tweet tidak ditemukan dalam waktu yang ditentukan
Tidak ada tweet yang ditemukan


In [8]:
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Fungsi Google Search menggunakan BeautifulSoup
def google_search_bs(query, num_results=10, num_pages=3):
    search_results = []
    base_url = "https://www.google.com/search"

    for page in range(num_pages):
        start = page * 50  # 10 results per page
        params = {
            'q': query,
            'start': start
        }
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36"
        }

        response = requests.get(base_url, headers=headers, params=params)
        soup = BeautifulSoup(response.text, 'html.parser')

        for g in soup.find_all('div', class_='g'):
            anchors = g.find_all('a')
            if anchors:
                link = anchors[0]['href']
                title = g.find('h3').text if g.find('h3') else 'No title'
                snippet = g.find('span', class_='aCOpRe').text if g.find('span', class_='aCOpRe') else 'No snippet'
                search_results.append({
                    'title': title,
                    'link': link,
                    'snippet': snippet
                })
    return search_results

# Fungsi untuk clustering menggunakan K-Means
def kmeans_clustering(snippets, n_clusters=3):
    # Menggunakan TF-IDF untuk mengubah teks menjadi representasi numerik
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(snippets)

    # Menggunakan K-Means untuk clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    labels = kmeans.fit_predict(X)

    # Menghitung silhouette score
    score = silhouette_score(X, labels)
    print(f"Silhouette Score: {score}")

    return labels

# Pencarian otomatis dan clustering menggunakan K-Means
def search_and_cluster(query):
    results = google_search_bs(query)
    snippets = [result['snippet'] for result in results if len(result['snippet'].split()) > 5]  # Hanya mengambil snippet dengan lebih dari 5 kata

    if not snippets:
        print("Tidak ada snippet yang cukup panjang untuk dianalisis.")
        return

    # Melakukan clustering pada snippets
    cluster_labels = kmeans_clustering(snippets, n_clusters=5)  # Mencoba 5 cluster

    # Menampilkan hasil pencarian dan cluster
    for idx, result in enumerate(results):
        print(f"Title: {result['title']}")
        print(f"URL: {result['link']}")
        print(f"Snippet: {result['snippet']}")
        print(f"Cluster: {cluster_labels[idx]}")  # Cluster label yang dihasilkan oleh K-Means
        print("-" * 50)

# Pencarian dan clustering terkait Shin Tae-yong
search_and_cluster("Shin Taeyong performa")


Tidak ada snippet yang cukup panjang untuk dianalisis.
